**Kinetic Rate and Superoperator Consistency Test**

**0. What is tested**

One test compares `EvolutionMatrix` with the population block of `EvolutionSuper`, including the resulting population kinetics. The existing sample/context generators, seven compositions and superoperator adapter are reused.

**1. Imports and settings**

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch

import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))

from sample_generators import make_sample_case

SEED = 42
DTYPE = torch.float64
DEVICE = torch.device("cpu")

from itertools import product
from context_generators import generate_contexts
from composer import compose_cases
from utils import get_superoperators
from mars.population import transform
from mars.population.tr_utils import EvolutionMatrix

FIELD = 0.34
TEMPERATURE = 3.0 
TIMES = torch.tensor([0.0, 1e-6, 1e-4, 1e-3, 1e-2], dtype=DTYPE, device=DEVICE)
RATE_ATOL = 1e-7
POPULATION_ATOL = 1e-7
RTOL = 1e-8


**2. Helper functions**

Evaluate the native rate interface in the same energy basis and powder orientation as `get_superoperators`. No rate matrix is reconstructed by hand.

In [2]:
def get_kinetic_matrix(sample, context, corrected: bool, orientation: int):
    """Build the native kinetic generator at the selected field and orientation."""
    F, _, _, Gz = sample.get_hamiltonian_terms()
    energies, vectors = torch.linalg.eigh(F + FIELD * Gz)
    if energies.ndim == 1:
        energies, vectors = energies.unsqueeze(0), vectors.unsqueeze(0)
    temperature = torch.tensor(TEMPERATURE, dtype=DTYPE, device=DEVICE)
    arguments = dict(
        full_system_vectors=vectors.unsqueeze(-3),
        fields=torch.full(energies.shape[:-1] + (1,), FIELD, dtype=DTYPE, device=DEVICE),
        energies=energies.unsqueeze(-2), temperature=temperature,
    )
    dimension = energies.shape[-1]

    def select(value, shape):
        return None if value is None else value.reshape(-1, *shape)[orientation].clone()

    evolution = EvolutionMatrix(
        energies[orientation], thermal_balance_mode="symmetric" if corrected else "skip"
    )
    return evolution(
        temperature,
        thermal_rates=select(context.get_transformed_thermal_rates(**arguments), (dimension, dimension)),
        driven_rates=select(context.get_transformed_driven_rates(**arguments), (dimension, dimension)),
        decay_rates=select(context.get_transformed_decay_rates(**arguments), (dimension,)),
    )


**3. Test**

Compare the complete generators, including their loss diagonals, then propagate the same normalized initial populations at every measurement time. Losses remain unnormalized during evolution.

In [3]:
def test_kinetic_superoperator_consistency(sample, context, *, secular, corrected, orientation):
    """Check rates and kinetics through the two native relaxation interfaces."""
    operators = get_superoperators(
        sample, context, magnetic_field_t=FIELD, temperature_k=TEMPERATURE,
        thermally_corrected=corrected, orientation=orientation,
    )
    kinetic = get_kinetic_matrix(sample, context, corrected, orientation)
    projected = transform.extract_transition_matrix_from_superoperator(operators["relaxation"])
    torch.testing.assert_close(kinetic.to(projected.dtype), projected, atol=RATE_ATOL, rtol=RTOL)

    populations = operators["initial_density"].diagonal().real
    populations = populations / populations.sum()
    expected = torch.matrix_exp(TIMES[:, None, None] * kinetic) @ populations
    actual = torch.matrix_exp(TIMES[:, None, None] * projected) @ populations.to(projected.dtype)
    torch.testing.assert_close(expected.to(actual.dtype), actual, atol=POPULATION_ATOL, rtol=RTOL)

    if secular:
        # In the secular case populations form a closed block of the relaxation superoperator. The diagonal Hamiltonian does not change populations.
        density = torch.diag(populations).to(operators["full"].dtype)
        propagators = transform.propagator_from_generator(operators["full"], TIMES)
        evolved = transform.Liouvilleator.unvec(propagators @ transform.Liouvilleator.vec(density))
        torch.testing.assert_close(
            expected, evolved.diagonal(dim1=-2, dim2=-1).real,
            atol=1e-6, rtol=RTOL,  # Full GHz-scale propagation has a larger roundoff error.
        )


**4. Test cases**

In [4]:
TEST_CASES = [
    {"spins": (0.5, 1.0), "partition": 1},
    {"spins": (0.5, 0.5, 1.0), "partition": 1},
    {"spins": (0.5, 0.5, 0.5, 0.5), "partition": 2},
]


**5. Run the test**

In [5]:
def run_tests(test_cases=TEST_CASES):
    """Run the one consistency test across all compositions and switches."""
    count = 0
    for parameters in test_cases:
        rng = np.random.default_rng(SEED)
        samples = make_sample_case(rng, **parameters, dtype=DTYPE, device=DEVICE, mesh=(5, 5))
        for secular, decay in product((False, True), repeat=2):
            contexts = generate_contexts(samples, rng, secular=secular, decay=decay, dtype=DTYPE, device=DEVICE)
            for composed in compose_cases(samples, contexts):
                F, _, _, _ = composed.sample.get_hamiltonian_terms()
                orientations = sorted({0, F.shape[0] - 1}) if F.ndim > 2 else [0]
                for corrected, orientation in product((False, True), orientations):
                    label = f"{composed.description}; secular={secular}, thermal={corrected}, decay={decay}, orientation={orientation}"
                    try:
                        test_kinetic_superoperator_consistency(
                            composed.sample, composed.context, secular=secular,
                            corrected=corrected, orientation=orientation,
                        )
                    except AssertionError as error:
                        raise AssertionError(f"{samples.description}; {label}\n{error}") from error
                    count += 1
        print(f"Kinetic/superoperator consistency passed ({samples.description})")
    print(f"\nOne test passed for all {count} parameter combinations.")


run_tests()


D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\spin_model.py:333: UserWarning: You are creating a block-diagonal (non-interacting) composite spin system via direct sum. This does NOT represent a true multi-particle quantum system (which would require a tensor-product space). Only use this if you are modeling effectively isolated subsystems (e.g., for polarized or time-resolved EPR). For physical spin clusters (diradicals, etc.), build a single SpinSystem with explicit couplings instead.
  concatenated_spin_system = concat_spin_systems(spin_systems)
D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\population\concatination.py:270: UserWarning: Contexts are combined by direct sum (block-diagonal composition). The subsystems evolve independently until transitions between them are added, e.g., mars.concat((context_1, context_2)) + context_exchange. Transitions between subsystems with different Hamiltonians are treated as incoherent population transfer; spectral (chemical) exchange is 

Kinetic/superoperator consistency passed (2 spins (0.5, 1.0), partition 1+1; Hilbert dimensions 2, 3, 6, 5; exchange MHz: J01=-81.8)


D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\spin_model.py:333: UserWarning: You are creating a block-diagonal (non-interacting) composite spin system via direct sum. This does NOT represent a true multi-particle quantum system (which would require a tensor-product space). Only use this if you are modeling effectively isolated subsystems (e.g., for polarized or time-resolved EPR). For physical spin clusters (diradicals, etc.), build a single SpinSystem with explicit couplings instead.
  concatenated_spin_system = concat_spin_systems(spin_systems)


Kinetic/superoperator consistency passed (3 spins (0.5, 0.5, 1.0), partition 1+2; Hilbert dimensions 2, 6, 12, 8; exchange MHz: J01=+93.9, J02=-12.4, J12=-96.0)
Kinetic/superoperator consistency passed (4 spins (0.5, 0.5, 0.5, 0.5), partition 2+2; Hilbert dimensions 4, 4, 16, 8; exchange MHz: J01=+31.7, J02=-60.0, J03=+72.1, J12=-32.0, J13=+61.2, J23=-100.4)

One test passed for all 336 parameter combinations.
